# CICLO DE ACOPLAMIENTO (GWF,GWT,WMA)

El siguiente script configura un ciclo de acoplamiento entre MODFLOW 6 (**GWF-GWT**) y un programa externo (**main_interfaz_equi.exe**), comenzando con la preparación de archivos iniciales y funciones auxiliares; en cada iteración, ejecuta los notebooks de flujo y transporte, obtiene la concentración final del modelo GWT y actualiza varios archivos (conc_i.m3d, u_tilde.dat y u_new1.out), ejecuta el programa main_interfaz_equi.exe para modificar u_new.out, reconstruye u_new1.out con información del tercer renglón, y al final transforma sus 30 valores en un archivo conc_i.m3d en formato de columna; además, proporciona una interfaz con slider y botón para controlar cuántas iteraciones ejecutar y mostrar el progreso en pantalla.

1. Se inicializan archivos y funciones para manejar conc_i.m3d, u_tilde.dat y u_new1.out.

2. El usuario fija cuántas iteraciones desea ejecutar mediante un slider y un botón.

3. En cada iteración, el script corre los notebooks de flujo y transporte (MF6).

4. Luego guarda la concentración final de GWT en conc_i.m3d como valores en columna.

5. Actualiza u_tilde.dat y ejecuta main_interfaz_equi.exe para generar un nuevo u_new.out.

6. Finalmente convierte u_new1.out en columna y lo guarda como nueva conc_i.m3d para la siguiente iteración.

7. El ciclo para este caso está constituído por los puntos (2,3,4,5,6).


# CICLO-GWF-GWT-WMA VERSION 1.1

In [ ]:
#******************************************************************************************************
# 1. Importación de librerías
import os; ruta_actual = os.getcwd() 
import shutil
import numpy as np
import subprocess
import time
import ipywidgets as widgets
from IPython.display import display
from IPython import get_ipython   # <--- IMPORTANTE para poder usar 'run' dentro de la función

# 2. Rutas de archivos
entrada_txt = "u_new.out"     # u_new.out
salida_m3d = "u_tilde.dat"    # u_tilde.dat
main_interfaz = "main_interfaz_equi.exe" 

shutil.copyfile('u_new0.out', 'conc_i.m3d')
shutil.copyfile('u_new0.out', 'u_new1.out')

#################################################################################
# 2.1  Sustituir primera línea de u_tilde.dat con TODO conc_i.m3d (aplanado)
def sustituir_primera_linea(archivo_u_tilde="u_tilde.dat", archivo_conc="conc_i.m3d"):
    try:
        # Lee conc_i.m3d y colapsa TODOS los valores (en columna) a un solo renglón
        with open(archivo_conc, 'r') as f_conc:
            tokens = []
            for line in f_conc:
                s = line.strip()
                if s:  # ignora líneas vacías
                    tokens.extend(s.split())
        if not tokens:
            print(f"⚠️ {archivo_conc} no contiene valores.")
            return
        linea_conc = " ".join(tokens) + "\n"

        # Lee u_tilde.dat
        with open(archivo_u_tilde, 'r') as f_ut:
            lineas = f_ut.readlines()

        if len(lineas) < 2:
            print(f"⚠️ El archivo {archivo_u_tilde} no tiene al menos dos líneas.")
            return

        # Sustituye SOLO el primer renglón por los valores en una sola línea
        lineas[0] = linea_conc

        # Escribe de vuelta (dejando el 2° renglón intacto)
        with open(archivo_u_tilde, 'w') as f_ut:
            f_ut.writelines(lineas)

        print(f"✅ Sustitución 1er renglón de '{archivo_u_tilde}' ← '{archivo_conc}' (columna→1 línea)")
        # print(linea_conc)  # opcional
    except Exception as e:
        print(f"❌ Error sustituyendo primera línea: {e}")

#################################################################################
# 2.2  Extrae el 3er renglón de u_new.out y lo pone como 1er renglón de u_new1.out
def extraer_tercer_renglon_u_new(u_new="u_new.out", destino="u_new1.out"):
    """
    Toma el 3er renglón de u_new y lo coloca como PRIMER renglón de 'destino' (u_new1.out),
    preservando el resto de líneas de destino.
    """
    try:
        # Lee u_new.out
        with open(u_new, "r") as f_src:
            lineas_src = f_src.readlines()
        if len(lineas_src) < 3:
            print(f"⚠️ {u_new} no tiene al menos 3 líneas.")
            return
        tercera = lineas_src[2].rstrip("\n") + "\n"

        # Lee (o crea) destino
        try:
            with open(destino, "r") as f_dst:
                lineas_dst = f_dst.readlines()
        except FileNotFoundError:
            lineas_dst = []

        if len(lineas_dst) == 0:
            # si no existe o está vacío, crea con una línea y nada más
            lineas_dst = [tercera]
        else:
            # sustituye solo la primera línea
            lineas_dst[0] = tercera

        # Escribe de vuelta
        with open(destino, "w") as f_dst:
            f_dst.writelines(lineas_dst)

        print(f"✅ 1er renglón de '{destino}' actualizado con el 3er renglón de '{u_new}'")

    except Exception as e:
        print(f"❌ Error sustituyendo renglón: {e}")

##################################################################################
# 3. Slider para controlar el ciclo_nv(b)
nv_slider = widgets.IntSlider(value=1, min=1, max=40, step=1, description="nv")
run_button = widgets.Button(description="Iniciar ciclo ")
output_area = widgets.Output()
contador_ejecuciones = 0

##################################################################################
# 4. Función que ejecuta el ciclo
def ciclo_nv(b):
    global contador_ejecuciones
    output_area.clear_output()
    
    with output_area:
        ip = get_ipython()   # <- Obtenemos el objeto IPython para usar 'run'
        contador_ejecuciones += 1
        print(f"🟢🟢🟢  Ejecución #{contador_ejecuciones}")        
        nv = nv_slider.value
        print(f"nv = {nv} iteraciones dentro de esta ejecución.\n")

        for i in range(nv):
            print(f"\n--- Iteración {i+1} de {nv} ---")
            
            # 4.1 Ejecutar flujo y transporte (MF6)
            print("▶️ Ejecutando GWF: 02_GWF.ipynb ...")
            ip.run_line_magic('run', '02_GWF.ipynb')

            print("▶️ Ejecutando GWT: 02_GWT.ipynb ...")
            ip.run_line_magic('run', '02_GWT.ipynb')

            # 4.2 Guardar concentración resultante de 02_transp_nvg.ipynb cobj -> conc_i.m3d
            #     (cobj -> conc (tiempo final) -> conc_flat -> conc_i.m3d, 1 valor por línea)
            try:
                # Obtener objeto de concentración desde o_gwt
                cobj = o_gwt.output.concentration()

                # Obtener todos los tiempos guardados
                times = np.array(cobj.get_times())
                print("   Tiempos disponibles en GWT:", times)

                if times.size == 0:
                    print("⚠️ No hay tiempos en la salida de GWT; no se actualiza conc_i.m3d.")
                else:
                    # Usar el último tiempo
                    t_final = times[-1]
                    print("   Usando el tiempo final:", t_final)

                    # Obtener matriz de concentración (nlay, nrow, ncol)
                    conc = cobj.get_data(totim=t_final)

                    # Aplanar a vector 1D (columna)
                    conc_flat = conc.ravel()

                    # Guardar en conc_i.m3d (SOLO 1 número por línea)
                    with open("conc_i.m3d", "w") as fout:
                        for val in conc_flat:
                            fout.write(f"{val}\n")

                    print(f"✅✅ conc_i.m3d creado con {conc_flat.size} valores del tiempo final {t_final}.")
                    print("   Concentraciones resultantes del tiempo final de 02_transp_nvg.ipynb (primeros 10):")
                    print("   conc_i.m3d:", conc_flat[:])
            except Exception as e:
                print(f"❌ Error al obtener cobj o guardar conc_i.m3d desde GWT: {e}")

#############################################################################################
            # 4.3 Actualizar u_tilde.dat a partir de conc_i.m3d
            print("▶️ Actualizando u_tilde.dat con conc_i.m3d ...")
            sustituir_primera_linea(archivo_u_tilde="u_tilde.dat", archivo_conc="conc_i.m3d")

#############################################################################################
            # 4.4 Ejecutar main_interfaz_equi.exe
            try:
                print(f"\n✅ Ejecutando {main_interfaz} ...\n")
                proceso = subprocess.Popen(
                    [main_interfaz],
                    stdout=subprocess.PIPE,
                    stderr=subprocess.STDOUT,
                    text=True
                )

                for linea in proceso.stdout:
                    print(linea, end="")  # salida en tiempo real

                proceso.wait()
                if proceso.returncode != 0:
                    print(f"❌ {main_interfaz} terminó con error. Código: {proceso.returncode}")
                    return
            except Exception as e:
                print(f"❌ Error al ejecutar {main_interfaz}: {e}")
                return

            # 4.5 Construir u_new1.out desde u_new.out copiando el 3er renglón al 1er renglón
            #print("\n▶️ Copiando el 3er renglón de u_new.out al 1er renglón de u_new1.out ...")
            extraer_tercer_renglon_u_new(u_new="u_new.out", destino="u_new1.out")            
           
            print("\n✅ Fin de la iteración, conc_i.m3d lista para próxima ejecución de GWT.")
            print("⏸️ Pausa de 1 segundo...\n")
            time.sleep(1)
        
##################################################################################
# 5. Conectar botón
run_button.on_click(ciclo_nv)

# extraer_tercer_renglon_u_new(u_new="u_new.out", destino="conc_i.m3d")
# Ahora: copiar u_new1.out → conc_i.m3d pero convertiendo las 30 columnas en 30 renglones
try:
    with open("u_new1.out", "r") as f_src:
        linea = f_src.readline().strip()   # Solo un renglón

    valores = linea.split()                # 30 columnas → lista de 30 strings

    with open("conc_i.m3d", "w") as f_dst:
        for v in valores:
            f_dst.write(v + "\n")          # Cada valor en su propio renglón

    print("📄 conc_i.m3d actualizado desde u_new1.out (30 columnas → 30 renglones).")

except Exception as e:
    print(f"❌ Error al convertir u_new1.out a conc_i.m3d en columna: {e}")

# 6. Mostrar interfaz
display(nv_slider, run_button, output_area)


# SECUENCIA DEL CICLO

**1. Importación de librerías**

Se cargan las librerías necesarias para manejar archivos, ejecutar MF6 y programas externos, controlar tiempos, crear interfaces con widgets y permitir la ejecución de notebooks desde Python mediante get_ipython().

**2. Preparación de archivos**

Se definen los nombres de los archivos clave (u_new.out, u_tilde.dat, main_interfaz_equi.exe)
y se inicializan dos archivos importantes copiando u_new0.out hacia:

conc_i.m3d (condición inicial del transporte),

u_new1.out (plantilla base para posteriores actualizaciones).

* 2.1 Función sustituir_primera_linea()

Convierte todo el contenido de conc_i.m3d (valores en columna) en un único renglón y reemplaza únicamente la primera línea de u_tilde.dat con ese renglón, conservando el resto del archivo. Reporta éxito o error.

* 2.2 Función extraer_tercer_renglon_u_new()

Lee el archivo u_new.out, extrae su tercer renglón y lo coloca como primer renglón de u_new1.out (creándolo o modificándolo). Reporta éxito o error.

**3. Widgets de control**

Se crea la interfaz:

un slider para seleccionar el número de iteraciones,

un botón para iniciar el ciclo,

un área de salida para mensajes,

un contador de ejecuciones.

**4. Función principal ciclo_nv()**

Cada vez que se presiona el botón:

Limpia la zona de mensajes.

Obtiene la consola IPython para ejecutar notebooks.

Incrementa el contador.

Lee cuántas iteraciones se deben ejecutar.

* En cada iteración:
* 4.1 Ejecuta flujo (GWF)

Lanza 02_flow_nvg.ipynb usando %run.

* 4.2 Ejecuta transporte (GWT)

Lanza 02_transp_nvg.ipynb usando %run.

* 4.3 Obtiene y guarda concentración

Lee la concentración final desde o_gwt, la aplana a un vector y la guarda en conc_i.m3d, un valor por renglón.

* 4.4 Actualiza u_tilde.dat

Convierte los valores de conc_i.m3d a un único renglón y sustituye la primera línea de u_tilde.dat.

* 4.5 Ejecuta main_interfaz_equi.exe

Corre el programa acoplador, muestra su salida, espera a que termine y detecta errores.

* 4.6 Construye u_new1.out

Toma el tercer renglón de u_new.out y lo inserta como primer renglón en u_new1.out.

* 4.7 Cierra la iteración

Imprime mensajes de fin de iteración y espera un segundo.

**5. Conversión final u_new1.out → conc_i.m3d**

Fuera del ciclo, se toma el renglón único de u_new1.out (30 columnas), se separa en valores individuales y se guarda en conc_i.m3d como 30 renglones (un valor por renglón). Se reporta éxito.

**6. Mostrar la interfaz**

Aparecen en pantalla el slider, el botón de ejecución y el área de mensajes, listos para iniciar el ciclo acoplado.